# GISLR — Landmark Motion-Over-Time Analysis

Diagnostic notebook (pre-`01_preprocess`, not part of the train pipeline). Measures
how much each landmark moves over time at three scopes: **per-video**, **per-category**,
**global**. See `TODO.md` §1 for the full spec.

**§1.0 decision (locked in):** DuckDB is the loading layer. Landmarks are queried
directly from parquet via a `CREATE VIEW ... glob` rather than materializing the
whole dataset in memory — aggregation happens before pulling into pandas, so peak
memory stays bounded by one query's result, not dataset size.

Sections:
1. Config & DuckDB connection (§1.1)
2. Reusable core: load / compute / plot (§1.1)
3. Resumable caching (§1.2)
4. Scope 1 — per-video (§1.3)
5. Scope 2 — per-category (§1.4)
6. Scope 3 — global (§1.5)
7. Cross-scope comparison (§1.6)

In [ ]:
# ============================================================
# CELL — Imports & config
# ============================================================
import json
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from tqdm.auto import tqdm

from modules.paths import DATASETS, CACHE_DIR

DATA_DIR: Path = DATASETS["GISLR"]
MOTION_CACHE_DIR: Path = CACHE_DIR / "motion_analysis"

SEED = 42
SG_WINDOW = 7
SG_POLYORDER = 2
LANDMARK_TYPE_ORDER: list[str] = ["pose", "left_hand", "right_hand", "face"]

rng: np.random.Generator = np.random.default_rng(SEED)

train_df: pd.DataFrame = pd.read_csv(DATA_DIR / "train.csv")
print(f"{len(train_df):,} sequences, {train_df['sign'].nunique()} signs")
train_df.head()

## 1. Reusable core (§1.1)

`get_duckdb_conn` → `load_landmarks_for_paths` → `compute_motion_energy` →
`plot_motion_gridspec`. All three scopes below call these same four functions;
nothing scope-specific lives inside them.

In [ ]:
# ============================================================
# CELL — get_duckdb_conn()
# ============================================================
def get_duckdb_conn(data_dir: Path) -> duckdb.DuckDBPyConnection:
    """In-memory DuckDB connection with a `meta_holistic` view joining
    train.csv metadata to the landmark parquet files via a glob pattern.
    Keeps peak memory bounded by one query's result, not the whole dataset
    (§1.0 — DuckDB accepted as the loading layer).
    """
    con = duckdb.connect(database=":memory:")
    glob_pattern = (data_dir / "train_landmark_files" / "*" / "*.parquet").as_posix()
    train_csv_path = (data_dir / "train.csv").as_posix()
    data_dir_str = data_dir.as_posix()
    backslash = chr(92)  # avoid ambiguous backslash-counting inside the f-string/SQL mix

    con.execute(f"""
        CREATE OR REPLACE VIEW meta_holistic AS
        WITH landmarks AS (
            SELECT
                -- normalize the OS-native separator to / FIRST (DuckDB returns
                -- filename with backslashes on Windows), THEN strip the GISLR
                -- root dir (already posix-form via .as_posix()) so this lines
                -- up with train.csv's relative "path" column for the join
                ltrim(
                    replace(replace(filename, '{backslash}', '/'), '{data_dir_str}', ''),
                    '/'
                ) AS rel_path,
                frame, type, landmark_index, x, y, z
            FROM read_parquet('{glob_pattern}', filename=true)
        )
        SELECT
            m.sign, m.participant_id, m.sequence_id, m.path,
            l.frame, l.type, l.landmark_index, l.x, l.y, l.z
        FROM landmarks l
        JOIN read_csv_auto('{train_csv_path}') m ON l.rel_path = m.path
    """)
    return con


con = get_duckdb_conn(DATA_DIR)
con.execute("SELECT COUNT(*) AS n_rows FROM meta_holistic").df()

In [ ]:
# ============================================================
# CELL — load_landmarks_for_paths()
# ============================================================
def load_landmarks_for_paths(
    con: duckdb.DuckDBPyConnection, paths: list[str]
) -> pd.DataFrame:
    """Single parameterized query — the one function all three scopes call.
    Memory-bounded: only rows for the requested `paths` are materialized.
    """
    query = """
        SELECT sign, participant_id, sequence_id, path, frame, type, landmark_index, x, y, z
        FROM meta_holistic
        WHERE path IN (SELECT UNNEST(?))
        ORDER BY path, frame
    """
    return con.execute(query, [paths]).df()


# smoke test on a single random path
_sample_path = train_df["path"].iloc[0]
load_landmarks_for_paths(con, [_sample_path]).head()

In [ ]:
# ============================================================
# CELL — compute_motion_energy()
# ============================================================
def compute_motion_energy(
    df: pd.DataFrame, window: int = SG_WINDOW, polyorder: int = SG_POLYORDER
) -> pd.DataFrame:
    """Group by ["path", "type", "landmark_index"] -> Savitzky-Golay smooth ->
    RMS speed (not mean-squared). Tidy long format:
    `path, video_id, sign, type, landmark_index, rms_speed`.

    Smoothing is done once per video across all landmarks simultaneously
    (one savgol_filter call over a (T, L*3) array) rather than looping per
    landmark — same result, far fewer calls. Looping stays per-video because
    the frame timeline (and the SG window) must never cross a video boundary.
    """
    rows = []
    for path, vid_df in df.groupby("path", sort=False):
        vid_df = vid_df.sort_values("frame")

        pivot_x = vid_df.pivot_table(
            index="frame", columns=["type", "landmark_index"], values="x", aggfunc="first"
        )
        pivot_y = vid_df.pivot_table(
            index="frame", columns=["type", "landmark_index"], values="y", aggfunc="first"
        ).reindex(columns=pivot_x.columns)
        pivot_z = vid_df.pivot_table(
            index="frame", columns=["type", "landmark_index"], values="z", aggfunc="first"
        ).reindex(columns=pivot_x.columns)

        landmarks = pivot_x.columns  # MultiIndex of (type, landmark_index)
        n_frames = len(pivot_x)

        arr = np.stack(
            [pivot_x.to_numpy(), pivot_y.to_numpy(), pivot_z.to_numpy()], axis=-1
        )  # (T, L, 3)
        arr2d = arr.reshape(n_frames, -1)  # (T, L*3)
        # undetected-landmark frames are NaN; interpolate along time per column.
        # a landmark never detected in this video stays all-NaN -> rms_speed NaN.
        arr2d = pd.DataFrame(arr2d).interpolate(limit_direction="both").to_numpy()

        valid_window = window if window % 2 == 1 else window - 1
        if n_frames > valid_window >= polyorder + 1:
            smoothed = savgol_filter(arr2d, window_length=valid_window, polyorder=polyorder, axis=0)
        else:
            smoothed = arr2d  # too short to smooth meaningfully — use raw positions

        smoothed = smoothed.reshape(n_frames, -1, 3)
        velocity = np.diff(smoothed, axis=0)  # (T-1, L, 3) per-frame displacement
        speed = np.linalg.norm(velocity, axis=2)  # (T-1, L)
        with np.errstate(invalid="ignore"):
            rms_speed = np.sqrt(np.nanmean(speed**2, axis=0))  # (L,)

        sign = vid_df["sign"].iloc[0]
        video_id = vid_df["sequence_id"].iloc[0]
        for (ltype, lidx), r in zip(landmarks, rms_speed):
            rows.append(
                {
                    "path": path,
                    "video_id": video_id,
                    "sign": sign,
                    "type": ltype,
                    "landmark_index": int(lidx),
                    "rms_speed": float(r),
                }
            )
    return pd.DataFrame(rows)


# smoke test, reusing the single-video load from the previous cell
compute_motion_energy(load_landmarks_for_paths(con, [_sample_path])).head()

In [ ]:
# ============================================================
# CELL — plot_motion_gridspec()
# ============================================================
def plot_motion_gridspec(
    df: pd.DataFrame,
    title: str,
    value_col: str = "rms_speed",
    err_col: str | None = None,
    save_path: Path | None = None,
) -> plt.Figure:
    """Combined overview (pose -> left_hand -> right_hand -> face, in that
    fixed order) on top, one zoomed per-type panel below. Parameterized over
    `value_col`/`err_col` so the same function serves single-video
    (`rms_speed`), per-category, and global (`rms_speed_mean`/`rms_speed_std`)
    scopes without three separate plotting functions.
    """
    fig = plt.figure(figsize=(16, 8))
    gs = fig.add_gridspec(2, len(LANDMARK_TYPE_ORDER), height_ratios=[1.2, 1])

    ax_overview = fig.add_subplot(gs[0, :])
    offset = 0
    ticks, labels = [], []
    for ltype in LANDMARK_TYPE_ORDER:
        sub = df[df["type"] == ltype].sort_values("landmark_index")
        x = offset + np.arange(len(sub))
        yerr = sub[err_col].to_numpy() if err_col and err_col in sub else None
        ax_overview.errorbar(
            x, sub[value_col], yerr=yerr, fmt="-o", markersize=2, elinewidth=0.5, label=ltype
        )
        if len(sub):
            ticks.append(offset + len(sub) / 2)
            labels.append(ltype)
        offset += len(sub)
    ax_overview.set_xticks(ticks)
    ax_overview.set_xticklabels(labels)
    ax_overview.set_ylabel(value_col)
    ax_overview.set_title(f"{title} — combined overview")
    ax_overview.legend(loc="upper right", fontsize=8)

    for i, ltype in enumerate(LANDMARK_TYPE_ORDER):
        ax = fig.add_subplot(gs[1, i])
        sub = df[df["type"] == ltype].sort_values("landmark_index")
        yerr = sub[err_col].to_numpy() if err_col and err_col in sub else None
        ax.errorbar(
            sub["landmark_index"], sub[value_col], yerr=yerr, fmt="-o", markersize=2, elinewidth=0.5
        )
        ax.set_title(ltype, fontsize=10)
        ax.set_xlabel("landmark_index")

    fig.suptitle(title)
    fig.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=120)
    return fig


# smoke test
_fig = plot_motion_gridspec(
    compute_motion_energy(load_landmarks_for_paths(con, [_sample_path])),
    title=f"smoke test — {_sample_path}",
)
plt.show()

## 2. Resumable caching / state management (§1.2)

One manifest per scope at `cache/motion_analysis/<scope>_manifest.json`, one
cached result file per unit at `cache/motion_analysis/<scope>/<id>.parquet`.
`process_units` is the shared driver: write the per-unit file **before**
marking it `done` in the manifest (a crash mid-write never leaves a `done`
entry pointing at a corrupt file), skip ids already `done` on resume, retry
`failed`, and never let one unit's exception kill the run.

In [ ]:
# ============================================================
# CELL — manifest helpers
# ============================================================
def _manifest_path(scope: str) -> Path:
    return MOTION_CACHE_DIR / f"{scope}_manifest.json"


def _unit_output_path(scope: str, unit_id: str) -> Path:
    return MOTION_CACHE_DIR / scope / f"{unit_id}.parquet"


def load_manifest(scope: str) -> dict:
    p = _manifest_path(scope)
    if p.exists():
        return json.loads(p.read_text())
    return {}


def save_manifest(scope: str, manifest: dict) -> None:
    p = _manifest_path(scope)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(manifest, indent=2))


def _mark(manifest: dict, scope: str, unit_id: str, status: str, artifact_path: Path | None = None) -> None:
    manifest[unit_id] = {
        "status": status,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "artifact_path": str(artifact_path) if artifact_path else manifest.get(unit_id, {}).get("artifact_path"),
    }
    save_manifest(scope, manifest)


def process_units(scope: str, unit_ids: list[str], process_fn) -> dict:
    """Shared load -> compute -> cache driver for every scope below.
    `process_fn(unit_id) -> pd.DataFrame` does the actual work; this function
    only owns resumability (skip done / retry failed), atomic-ish writes
    (file written before the manifest says `done`), and failure isolation.
    """
    manifest = load_manifest(scope)
    out_dir = MOTION_CACHE_DIR / scope
    out_dir.mkdir(parents=True, exist_ok=True)

    todo = [u for u in unit_ids if manifest.get(u, {}).get("status") != "done"]
    for unit_id in tqdm(todo, desc=f"[{scope}] processing"):
        try:
            result_df = process_fn(unit_id)
            out_path = _unit_output_path(scope, unit_id)
            result_df.to_parquet(out_path)  # write BEFORE marking done
            _mark(manifest, scope, unit_id, "done", out_path)
        except Exception as e:
            _mark(manifest, scope, unit_id, "failed")
            tqdm.write(f"[{scope}] FAILED {unit_id}: {e}")
    return manifest


def load_cached_units(scope: str, unit_ids: list[str]) -> pd.DataFrame:
    """Final aggregation reads only cached per-unit files, never recomputes
    from raw parquet."""
    frames = [pd.read_parquet(_unit_output_path(scope, u)) for u in unit_ids]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 3. Scope 1 — per-video, 50 random samples (§1.3)

Unit id = `sequence_id` (unique per video, filename-safe). Output:
`cache/motion_analysis/per_video/summary.parquet`
(`video_id, sign, type, landmark_index, rms_speed`). Per-video plots are
saved as PNGs under `cache/motion_analysis/per_video/plots/`.

In [ ]:
# ============================================================
# CELL — Scope 1: sample 50 videos (fixed seed, recorded for reproducibility)
# ============================================================
N_VIDEOS = 50
per_video_sample = train_df.sample(n=N_VIDEOS, random_state=SEED).reset_index(drop=True)
per_video_sample["sequence_id"] = per_video_sample["sequence_id"].astype(str)

print(f"Scope 1 — per-video sample: seed={SEED}, n={len(per_video_sample)}")
per_video_sample[["sequence_id", "sign", "path"]].head()